# 06 - Chronos-2 Experimental Forecasts

This notebook runs a focused Chronos-2 experiment ladder for Danish day-ahead electricity prices:

1. zero-shot, target only, context 336
2. zero-shot, target only, context 1344
3. zero-shot, calendar covariates, context 1344
4. zero-shot, calendar + weather covariates, context 1344
5. LoRA, calendar + weather covariates, context 1024, 300 steps

The implementation uses Amazon's `chronos-forecasting` package for `amazon/chronos-2`. That package wraps the Hugging Face Transformers model and exposes the Chronos-2 pandas API (`predict_df`) plus `fit(..., finetune_mode="lora")` for adapter tuning.

<p style="color:#b00020"><strong>Interpretation guide.</strong> Chronos predicts the next contiguous timestamps after its context. This repo's production-style origin forecasts tomorrow's Danish local delivery day, so each origin forecasts the bridge after the forecast origin plus the delivery day, then scores only delivery-day rows. The default validation arena is the same compact recent weather arena used by the CatBoost development notebook: 2026-04 through 2026-06, with earlier weather-frame context retained for tuning comparisons.</p>

## 1. Setup And Experiment Contract

The notebook keeps all Chronos-specific orchestration local. It reuses the repo's panel loader, Danish delivery-day horizon builder, evaluation tables, availability-safe weather joins, CatBoost weather validation origins, and production baseline comparator.

On a fresh environment, install the extra dependencies before loading the model:

```python
%pip install "chronos-forecasting[extras]>=2.2" "peft>=0.11" "accelerate>=0.30"
```

GPU is strongly recommended for the LoRA run. CPU inference is supported, but the covariate experiments and 300-step adapter tuning will be slow. On Apple Silicon, the notebook defaults to MPS when PyTorch exposes it; set `CHRONOS_SMOKE_MODE = True` for a two-origin, two-step mechanics check.

In [ ]:
from __future__ import annotations

import json
import math
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_forecast.backtesting.horizons import (
    make_daily_origins,
    make_danish_delivery_day_horizon,
)
from dkenergy_forecast.evaluation.summary import (
    add_prediction_diagnostics,
    model_score_table,
    probabilistic_metric_table,
)
from dkenergy_forecast.features.weather_features import (
    add_weather_derived_features,
    add_weather_ensemble_features,
    build_weather_experiment_frame,
    join_weather_features,
    weather_value_columns,
)
from dkenergy_forecast.io import load_price_panel
from dkenergy_forecast.tuning import baseline_predictions_from_feature_frame
from dkenergy_forecast.tuning.catboost_validation import outer_origins_for_months, prepare_nested_frame
from dkenergy_forecast.types import add_copenhagen_calendar, normalize_utc_column, to_utc_timestamp

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

warnings.filterwarnings("once", category=UserWarning)

In [ ]:
PANEL_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.parquet"
QA_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.qa.json"
WEATHER_LONG_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "weather_open_meteo_area_hourly_long_open_meteo_previous_runs_v1.parquet"
)
WEATHER_FRAME_PATH = PROJECT_ROOT / "data" / "features" / "weather_experiment_frame_backtest.parquet"
RESULT_DIR = PROJECT_ROOT / "results" / "notebook_chronos2_experimental_v1"

MODEL_ID = "amazon/chronos-2"

def default_chronos_device_map() -> str:
    if torch.cuda.is_available():
        return "cuda"
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return "mps"
    return "cpu"

DEVICE_MAP: str | None = default_chronos_device_map()
AT_HOUR_UTC = 10
MIN_HISTORY_DAYS = 365
WEATHER_FRAME_MONTHS = tuple(str(period) for period in pd.period_range("2025-11", "2026-06", freq="M"))
WEATHER_BACKTEST_MONTHS = tuple(str(period) for period in pd.period_range("2026-04", "2026-06", freq="M"))
CHRONOS_SMOKE_MODE = False
SMOKE_MAX_ORIGINS: int | None = None  # Set to a small integer for a quick mechanics run.

QUANTILE_LEVELS = [0.1, 0.5, 0.9]
PREDICTION_BATCH_SIZE = 128
CROSS_LEARNING = False
RUN_ZERO_SHOT = True
RUN_LORA = True
WRITE_ARTIFACTS = True

CALENDAR_COVARIATES = [
    "local_hour",
    "local_day_of_week",
    "local_month",
    "is_weekend",
    "is_dst",
    "utc_offset_hours",
]
WEATHER_COVARIATE_MODE: Literal["all", "raw", "ensemble", "ensemble_mean"] = "all"
ADD_WEATHER_ENSEMBLE_FEATURES = True
ADD_WEATHER_DERIVED_FEATURES = False

LORA_CONTEXT_LENGTH = 1024
LORA_NUM_STEPS = 300
LORA_BATCH_SIZE = 32
LORA_LEARNING_RATE = 1e-4
LORA_LOGGING_STEPS = 25
LORA_TRAIN_DAYS = 180
LORA_MAX_TRAIN_ORIGINS = 90
LORA_PREDICTION_LENGTH: int | None = None  # None uses the max bridge-plus-delivery length in scored origins.

if CHRONOS_SMOKE_MODE:
    SMOKE_MAX_ORIGINS = 2
    RUN_ZERO_SHOT = True
    RUN_LORA = True
    WRITE_ARTIFACTS = False
    PREDICTION_BATCH_SIZE = 8
    WEATHER_COVARIATE_MODE = "ensemble_mean"
    ADD_WEATHER_ENSEMBLE_FEATURES = True
    ADD_WEATHER_DERIVED_FEATURES = False
    LORA_CONTEXT_LENGTH = 336
    LORA_NUM_STEPS = 2
    LORA_BATCH_SIZE = 1
    LORA_LOGGING_STEPS = 1
    LORA_TRAIN_DAYS = 30
    LORA_MAX_TRAIN_ORIGINS = 4

LORA_TRAINER_KWARGS = {
    "output_dir": str(RESULT_DIR / "chronos2_lora_trainer"),
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": DEVICE_MAP != "mps",
}

@dataclass(frozen=True)
class ChronosExperiment:
    name: str
    description: str
    covariate_set: Literal["target_only", "calendar", "calendar_weather"]
    context_length: int
    finetune_mode: Literal["zero_shot", "lora"] = "zero_shot"
    num_steps: int | None = None

EXPERIMENTS = [
    ChronosExperiment(
        name="chronos2_zs_target_ctx336",
        description="zero-shot, target only, context 336",
        covariate_set="target_only",
        context_length=336,
    ),
    ChronosExperiment(
        name="chronos2_zs_target_ctx1344",
        description="zero-shot, target only, context 1344",
        covariate_set="target_only",
        context_length=1344,
    ),
    ChronosExperiment(
        name="chronos2_zs_calendar_ctx1344",
        description="zero-shot, calendar covariates, context 1344",
        covariate_set="calendar",
        context_length=1344,
    ),
    ChronosExperiment(
        name="chronos2_zs_calendar_weather_ctx1344",
        description="zero-shot, calendar + weather covariates, context 1344",
        covariate_set="calendar_weather",
        context_length=1344,
    ),
    ChronosExperiment(
        name="chronos2_lora_calendar_weather_ctx1024_300steps",
        description="LoRA, calendar + weather covariates, context 1024, 300 steps",
        covariate_set="calendar_weather",
        context_length=LORA_CONTEXT_LENGTH,
        finetune_mode="lora",
        num_steps=LORA_NUM_STEPS,
    ),
]

if CHRONOS_SMOKE_MODE:
    EXPERIMENTS = [
        ChronosExperiment(
            name="smoke_chronos2_zs_target_ctx336",
            description="smoke, zero-shot, target only, context 336",
            covariate_set="target_only",
            context_length=336,
        ),
        ChronosExperiment(
            name="smoke_chronos2_lora_calendar_weather_ctx336_2steps",
            description="smoke, LoRA, calendar + weather covariates, context 336, 2 steps",
            covariate_set="calendar_weather",
            context_length=LORA_CONTEXT_LENGTH,
            finetune_mode="lora",
            num_steps=LORA_NUM_STEPS,
        ),
    ]

RESULT_DIR

## 2. Load Data And Shared Validation Origins

The default origin set is the CatBoost notebook's compact recent weather validation arena, `2026-04` through `2026-06`, with `2025-11` through `2026-03` retained as weather-frame context before the first scored block. This is a development validation arena, not a final untouched test set: it is fair for deciding whether Chronos is worth pursuing, but optimistic if repeatedly used to crown a production winner.

In [ ]:
def require_months(frame: pd.DataFrame, months: tuple[str, ...], label: str) -> None:
    missing = sorted(set(months) - set(frame["outer_month"].dropna().unique()))
    if missing:
        raise ValueError(f"{label} is missing requested month(s): {missing}")


def month_span_origins(panel: pd.DataFrame, months: tuple[str, ...], *, at_hour_utc: int = AT_HOUR_UTC) -> pd.DataFrame:
    start = pd.Period(months[0], freq="M").to_timestamp().tz_localize("UTC")
    end = (pd.Period(months[-1], freq="M") + 1).to_timestamp().tz_localize("UTC")
    return make_daily_origins(panel, start=start, end=end, at_hour_utc=at_hour_utc)


def build_or_load_weather_validation_frame(panel: pd.DataFrame, months: tuple[str, ...]) -> pd.DataFrame:
    if WEATHER_FRAME_PATH.exists():
        cached = prepare_nested_frame(pd.read_parquet(WEATHER_FRAME_PATH))
        missing = sorted(set(months) - set(cached["outer_month"].dropna().unique()))
        if not missing:
            return cached
        print(f"Cached weather validation frame is missing context month(s) {missing}; rebuilding {WEATHER_FRAME_PATH}.")

    if not WEATHER_LONG_PATH.exists():
        raise FileNotFoundError(
            f"Missing {WEATHER_LONG_PATH}. Build Open-Meteo weather features before running this notebook."
        )
    weather_long_source = pd.read_parquet(WEATHER_LONG_PATH)
    frame_origins = month_span_origins(panel, months)
    output = build_weather_experiment_frame(panel, frame_origins, weather_long_source)
    WEATHER_FRAME_PATH.parent.mkdir(parents=True, exist_ok=True)
    output.to_parquet(WEATHER_FRAME_PATH, index=False)
    return prepare_nested_frame(output)


if not WEATHER_LONG_PATH.exists():
    raise FileNotFoundError(
        f"Missing {WEATHER_LONG_PATH}. Build Open-Meteo weather features before running this notebook."
    )

panel = load_price_panel(
    PANEL_PATH,
    QA_PATH if QA_PATH.exists() else None,
    require_final_historical=False,
)
weather_frame = build_or_load_weather_validation_frame(panel, WEATHER_FRAME_MONTHS)
require_months(weather_frame, WEATHER_FRAME_MONTHS, "weather validation frame")
origins = outer_origins_for_months(weather_frame, WEATHER_BACKTEST_MONTHS)
if SMOKE_MAX_ORIGINS is not None:
    origins = origins.tail(SMOKE_MAX_ORIGINS).reset_index(drop=True)
weather_long = pd.read_parquet(WEATHER_LONG_PATH)

coverage = {
    "panel_rows": len(panel),
    "panel_min_ds_utc": panel["ds_utc"].min(),
    "panel_max_ds_utc": panel["ds_utc"].max(),
    "origin_count": len(origins),
    "origin_min_utc": origins["forecast_origin_utc"].min(),
    "origin_max_utc": origins["forecast_origin_utc"].max(),
    "validation_arena": "catboost_weather_overlap",
    "validation_months": WEATHER_BACKTEST_MONTHS,
    "weather_frame_months": WEATHER_FRAME_MONTHS,
    "weather_frame_rows": len(weather_frame),
    "weather_rows": len(weather_long),
    "smoke_max_origins": SMOKE_MAX_ORIGINS,
}
coverage

In [ ]:
origin_horizon_summary = []
for origin in origins["forecast_origin_utc"]:
    horizon = make_danish_delivery_day_horizon(panel, origin, days_ahead=1)
    history_last = panel.loc[panel["ds_utc"] < origin, "ds_utc"].max()
    prediction_length = int((horizon["ds_utc"].max() - history_last) / pd.Timedelta(hours=1))
    origin_horizon_summary.append(
        {
            "forecast_origin_utc": origin,
            "history_last_utc": history_last,
            "delivery_start_utc": horizon["ds_utc"].min(),
            "delivery_end_utc": horizon["ds_utc"].max(),
            "delivery_rows": len(horizon),
            "chronos_prediction_length": prediction_length,
        }
    )
origin_horizon_summary = pd.DataFrame(origin_horizon_summary)
effective_lora_prediction_length = (
    int(LORA_PREDICTION_LENGTH)
    if LORA_PREDICTION_LENGTH is not None
    else int(origin_horizon_summary["chronos_prediction_length"].max())
)
display({"effective_lora_prediction_length": effective_lora_prediction_length})
origin_horizon_summary

## 3. Chronos Frame Builders

`predict_df` forecasts a contiguous future range. For each origin, the helper below builds:

- a Chronos context frame ending at `forecast_origin_utc - 1h`
- an optional future covariate frame covering the bridge plus the delivery day
- delivery-day metadata for scoring only the intended day-ahead horizon

In [ ]:
@dataclass
class ChronosOriginFrames:
    origin: pd.Timestamp
    context_df: pd.DataFrame
    future_df: pd.DataFrame | None
    horizon_metadata: pd.DataFrame
    prediction_length: int
    covariates: list[str]
    history_full: pd.DataFrame
    future_full: pd.DataFrame


def to_chronos_timestamp(series: pd.Series) -> pd.Series:
    values = pd.to_datetime(series, utc=True)
    return values.dt.tz_localize(None)


def from_chronos_timestamp(series: pd.Series) -> pd.Series:
    values = pd.to_datetime(series)
    if values.dt.tz is None:
        return values.dt.tz_localize("UTC")
    return values.dt.tz_convert("UTC")


def normalize_covariate_dtypes(frame: pd.DataFrame, covariates: list[str]) -> pd.DataFrame:
    output = frame.copy()
    for column in covariates:
        if column not in output.columns:
            output[column] = pd.NA
        if pd.api.types.is_bool_dtype(output[column]):
            output[column] = output[column].astype("int8")
    return output


def selected_weather_columns(frame: pd.DataFrame, mode: str = WEATHER_COVARIATE_MODE) -> list[str]:
    columns = weather_value_columns(frame)
    if mode == "raw":
        columns = [column for column in columns if not column.startswith("weather_ensemble_")]
    elif mode == "ensemble":
        columns = [column for column in columns if column.startswith("weather_ensemble_")]
    elif mode == "ensemble_mean":
        columns = [
            column
            for column in columns
            if column.startswith("weather_ensemble_") and column.endswith("_mean")
        ]
    elif mode != "all":
        raise ValueError(f"Unknown WEATHER_COVARIATE_MODE: {mode!r}")
    return columns


def add_weather_covariates(frame: pd.DataFrame, weather: pd.DataFrame | None) -> pd.DataFrame:
    if weather is None:
        raise FileNotFoundError(
            f"Missing {WEATHER_LONG_PATH}. Build weather features before running calendar+weather experiments."
        )
    enriched = join_weather_features(frame, weather)
    if ADD_WEATHER_ENSEMBLE_FEATURES:
        enriched = add_weather_ensemble_features(enriched)
    if ADD_WEATHER_DERIVED_FEATURES:
        enriched = add_weather_derived_features(enriched)
    return enriched


def make_future_base(panel_utc: pd.DataFrame, origin: pd.Timestamp, prediction_timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    ids = panel_utc[["unique_id", "area", "dataset_version"]].drop_duplicates("unique_id")
    future = ids.merge(pd.DataFrame({"ds_utc": prediction_timestamps}), how="cross")
    future["forecast_origin_utc"] = origin
    future = add_copenhagen_calendar(future)
    return future.sort_values(["unique_id", "ds_utc"]).reset_index(drop=True)


def make_horizon_metadata(panel_utc: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    horizon = make_danish_delivery_day_horizon(panel_utc, origin, days_ahead=1)
    actual_columns = [
        "unique_id",
        "ds_utc",
        "area",
        "ds_local",
        "local_date",
        "local_hour",
        "local_day_of_week",
        "local_month",
        "is_weekend",
        "is_dst",
        "utc_offset_hours",
        "dataset_version",
        "y",
    ]
    actuals = panel_utc[actual_columns].drop_duplicates(["unique_id", "ds_utc"])
    metadata = horizon[["unique_id", "ds_utc", "forecast_origin_utc", "horizon"]].merge(
        actuals,
        on=["unique_id", "ds_utc"],
        how="left",
    )
    return metadata.sort_values(["unique_id", "ds_utc"]).reset_index(drop=True)


def prepare_origin_frames(
    panel: pd.DataFrame,
    origin: object,
    *,
    context_length: int,
    covariate_set: Literal["target_only", "calendar", "calendar_weather"],
    weather: pd.DataFrame | None = None,
) -> ChronosOriginFrames:
    panel_utc = normalize_utc_column(panel, "ds_utc").sort_values(["unique_id", "ds_utc"])
    origin_utc = to_utc_timestamp(origin)

    history = panel_utc.loc[panel_utc["ds_utc"] < origin_utc].copy()
    history = history.groupby("unique_id", group_keys=False).tail(context_length)
    if history.empty:
        raise ValueError(f"No history available before {origin_utc}")

    history_lengths = history.groupby("unique_id")["ds_utc"].size()
    if int(history_lengths.min()) < context_length:
        raise ValueError(
            f"Insufficient history before {origin_utc}: minimum rows per series is {int(history_lengths.min())}"
        )

    last_by_id = history.groupby("unique_id")["ds_utc"].max()
    if last_by_id.nunique() != 1:
        raise ValueError(f"Series do not share the same final context timestamp before {origin_utc}")
    last_context_ts = last_by_id.iloc[0]

    horizon_metadata = make_horizon_metadata(panel_utc, origin_utc)
    prediction_timestamps = pd.date_range(
        start=last_context_ts + pd.Timedelta(hours=1),
        end=horizon_metadata["ds_utc"].max(),
        freq="h",
        tz="UTC",
    )
    prediction_length = len(prediction_timestamps)

    history_full = history.copy()
    history_full["forecast_origin_utc"] = origin_utc
    future_full = make_future_base(panel_utc, origin_utc, prediction_timestamps)

    if covariate_set == "calendar_weather":
        combined = pd.concat(
            [
                history_full.assign(_chronos_part="context"),
                future_full.assign(_chronos_part="future"),
            ],
            ignore_index=True,
            sort=False,
        )
        combined = add_weather_covariates(combined, weather)
        weather_covariates = selected_weather_columns(combined)
        history_full = combined.loc[combined["_chronos_part"].eq("context")].drop(columns="_chronos_part")
        future_full = combined.loc[combined["_chronos_part"].eq("future")].drop(columns="_chronos_part")
    else:
        weather_covariates = []

    covariates: list[str] = []
    if covariate_set in {"calendar", "calendar_weather"}:
        covariates.extend([column for column in CALENDAR_COVARIATES if column in history_full.columns])
    if covariate_set == "calendar_weather":
        covariates.extend(weather_covariates)

    covariates = [
        column
        for column in dict.fromkeys(covariates)
        if column in history_full.columns and column in future_full.columns
    ]
    covariates = [
        column
        for column in covariates
        if not pd.concat([history_full[column], future_full[column]], ignore_index=True).isna().all()
    ]

    history_full = normalize_covariate_dtypes(history_full, covariates)
    future_full = normalize_covariate_dtypes(future_full, covariates)

    context_columns = ["unique_id", "ds_utc", "y", *covariates]
    context_df = history_full[context_columns].rename(
        columns={"unique_id": "item_id", "ds_utc": "timestamp", "y": "target"}
    )
    context_df["timestamp"] = to_chronos_timestamp(context_df["timestamp"])
    context_df = context_df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

    future_df = None
    if covariates:
        future_columns = ["unique_id", "ds_utc", *covariates]
        future_df = future_full[future_columns].rename(
            columns={"unique_id": "item_id", "ds_utc": "timestamp"}
        )
        future_df["timestamp"] = to_chronos_timestamp(future_df["timestamp"])
        future_df = future_df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

    return ChronosOriginFrames(
        origin=origin_utc,
        context_df=context_df,
        future_df=future_df,
        horizon_metadata=horizon_metadata,
        prediction_length=prediction_length,
        covariates=covariates,
        history_full=history_full,
        future_full=future_full,
    )

In [ ]:
def rename_chronos_quantiles(predictions: pd.DataFrame) -> pd.DataFrame:
    output = predictions.copy()
    rename_map = {}
    for level in QUANTILE_LEVELS:
        quantile_name = f"q{int(round(level * 100))}"
        candidates = [str(level), f"{level:.1f}", f"{level:.2f}"]
        for candidate in candidates:
            if candidate in output.columns:
                rename_map[candidate] = quantile_name
                break
    output = output.rename(columns=rename_map)
    if "predictions" in output.columns:
        output["y_pred"] = output["predictions"]
    elif "q50" in output.columns:
        output["y_pred"] = output["q50"]
    else:
        raise ValueError("Chronos output did not include predictions or q50")
    return output


def predict_one_origin(
    pipeline,
    experiment: ChronosExperiment,
    panel: pd.DataFrame,
    origin: object,
    *,
    weather: pd.DataFrame | None = None,
) -> pd.DataFrame:
    frames = prepare_origin_frames(
        panel,
        origin,
        context_length=experiment.context_length,
        covariate_set=experiment.covariate_set,
        weather=weather,
    )
    pred_df = pipeline.predict_df(
        frames.context_df,
        future_df=frames.future_df,
        prediction_length=frames.prediction_length,
        quantile_levels=QUANTILE_LEVELS,
        id_column="item_id",
        timestamp_column="timestamp",
        target="target",
        batch_size=PREDICTION_BATCH_SIZE,
        context_length=experiment.context_length,
        cross_learning=CROSS_LEARNING,
        freq="h",
    )

    pred_df = rename_chronos_quantiles(pred_df)
    pred_df = pred_df.rename(columns={"item_id": "unique_id"})
    pred_df["ds_utc"] = from_chronos_timestamp(pred_df["timestamp"])

    prediction_columns = [
        "unique_id",
        "ds_utc",
        "y_pred",
        *[column for column in ["q10", "q50", "q90"] if column in pred_df.columns],
    ]
    scored = frames.horizon_metadata.merge(
        pred_df[prediction_columns],
        on=["unique_id", "ds_utc"],
        how="left",
    )
    scored["model_label"] = experiment.name
    scored["experiment_description"] = experiment.description
    scored["context_length"] = experiment.context_length
    scored["covariate_set"] = experiment.covariate_set
    scored["finetune_mode"] = experiment.finetune_mode
    scored["chronos_prediction_length"] = frames.prediction_length
    scored["covariate_count"] = len(frames.covariates)
    return scored


def run_chronos_backtest(
    pipeline,
    experiment: ChronosExperiment,
    panel: pd.DataFrame,
    origins: pd.DataFrame,
    *,
    weather: pd.DataFrame | None = None,
) -> pd.DataFrame:
    outputs = []
    origin_values = origins["forecast_origin_utc"].sort_values().drop_duplicates().tolist()
    for index, origin in enumerate(origin_values, start=1):
        print(f"[{index}/{len(origin_values)}] {experiment.name} @ {origin}")
        outputs.append(predict_one_origin(pipeline, experiment, panel, origin, weather=weather))
    if not outputs:
        return pd.DataFrame()
    return pd.concat(outputs, ignore_index=True)


def experiment_needs_weather(experiment: ChronosExperiment) -> bool:
    return experiment.covariate_set == "calendar_weather"


## 4. Baselines On The Same Origins

The CatBoost notebook treats production baselines as the hurdle. This notebook uses the same comparator on the same weather validation origins so Chronos variants are not evaluated in isolation.

In [ ]:
baseline_predictions = baseline_predictions_from_feature_frame(
    weather_frame,
    origins=origins,
)
baseline_scores = model_score_table(add_prediction_diagnostics(baseline_predictions))
baseline_scores[baseline_scores["area"].eq("ALL")].sort_values("mae")

## 5. Load Chronos-2

This cell downloads `amazon/chronos-2` the first time it runs. Keep `DEVICE_MAP = "cpu"` in the config if you want to force CPU; otherwise the notebook chooses CUDA, MPS, then CPU in that order.

In [ ]:
try:
    from chronos import BaseChronosPipeline, Chronos2Pipeline
except ImportError as exc:
    raise ImportError(
        'Chronos is not installed. Run `%pip install "chronos-forecasting[extras]>=2.2" "peft>=0.11"` first.'
    ) from exc

pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE_MAP,
)
pipeline

## 6. Zero-Shot Runs

These four experiments keep the pretrained model fixed. The only changes are context length and whether known-future calendar/weather covariates are included.

In [ ]:
zero_shot_predictions: list[pd.DataFrame] = []
zero_shot_experiments = [experiment for experiment in EXPERIMENTS if experiment.finetune_mode == "zero_shot"]

if RUN_ZERO_SHOT:
    for experiment in zero_shot_experiments:
        if experiment_needs_weather(experiment) and weather_long is None:
            print(f"Skipping {experiment.name}: weather features are missing")
            continue
        zero_shot_predictions.append(
            run_chronos_backtest(
                pipeline,
                experiment,
                panel,
                origins,
                weather=weather_long,
            )
        )
else:
    print("RUN_ZERO_SHOT is False")

zero_shot_predictions_df = (
    pd.concat(zero_shot_predictions, ignore_index=True) if zero_shot_predictions else pd.DataFrame()
)
zero_shot_predictions_df.shape

In [ ]:
if not zero_shot_predictions_df.empty:
    zero_shot_scores = model_score_table(add_prediction_diagnostics(zero_shot_predictions_df))
    display(zero_shot_scores[zero_shot_scores["area"].eq("ALL")].sort_values("mae"))
else:
    zero_shot_scores = pd.DataFrame()
    print("No zero-shot predictions yet.")

## 7. LoRA Fine-Tuning Data

Chronos-2's fine-tuning dataset samples contiguous windows from the provided series. To make the adapter train on the same operational task, each training item below is a fixed final window:

- `context_length` historical target/covariate rows ending one hour before the origin
- `prediction_length` future rows that include the origin-to-delivery bridge plus the delivery day
- `min_past=context_length` during `fit`, so the only sampled training slice is that final task window

`effective_lora_prediction_length` defaults to the maximum bridge-plus-delivery length in the scored origins. That keeps the adapter capable of the longest validation horizon while still allowing shorter zero-shot or LoRA prediction calls during evaluation. Training origins with a different bridge-plus-delivery length are skipped rather than mixed into a fixed-length fine-tuning dataset.

In [ ]:
def choose_lora_training_origins(
    panel: pd.DataFrame,
    evaluation_origins: pd.DataFrame,
    *,
    train_days: int,
    max_origins: int,
    at_hour_utc: int,
    min_history_days: int,
) -> pd.DataFrame:
    panel_utc = normalize_utc_column(panel, "ds_utc")
    first_eval_origin = to_utc_timestamp(evaluation_origins["forecast_origin_utc"].min())
    start = first_eval_origin - pd.Timedelta(days=train_days)
    earliest_allowed = panel_utc["ds_utc"].min() + pd.Timedelta(days=min_history_days)
    start = max(start, earliest_allowed)
    end = first_eval_origin.normalize()

    candidate_origins = make_daily_origins(panel_utc, start=start, end=end, at_hour_utc=at_hour_utc)
    valid: list[pd.Timestamp] = []
    for origin in candidate_origins["forecast_origin_utc"]:
        if origin >= first_eval_origin:
            continue
        horizon = make_danish_delivery_day_horizon(panel_utc, origin, days_ahead=1)
        if horizon.empty:
            continue
        if horizon["ds_utc"].max() <= panel_utc["ds_utc"].max():
            valid.append(origin)

    if max_origins > 0:
        valid = valid[-max_origins:]
    return pd.DataFrame({"forecast_origin_utc": valid})


def make_lora_training_frame(
    panel: pd.DataFrame,
    training_origins: pd.DataFrame,
    *,
    context_length: int,
    prediction_length: int,
    weather: pd.DataFrame | None,
) -> tuple[pd.DataFrame, list[str], pd.DataFrame]:
    panel_utc = normalize_utc_column(panel, "ds_utc")
    actuals = panel_utc[["unique_id", "ds_utc", "y"]].drop_duplicates(["unique_id", "ds_utc"])
    frames: list[pd.DataFrame] = []
    skipped: list[dict[str, object]] = []
    covariate_union: list[str] = []

    for origin in training_origins["forecast_origin_utc"].sort_values().drop_duplicates():
        try:
            origin_frames = prepare_origin_frames(
                panel_utc,
                origin,
                context_length=context_length,
                covariate_set="calendar_weather",
                weather=weather,
            )
        except Exception as exc:
            skipped.append({"forecast_origin_utc": origin, "reason": repr(exc)})
            continue

        if origin_frames.prediction_length != prediction_length:
            skipped.append(
                {
                    "forecast_origin_utc": origin,
                    "reason": f"prediction_length={origin_frames.prediction_length}",
                }
            )
            continue

        future_with_y = origin_frames.future_full.drop(columns=["y"], errors="ignore").merge(
            actuals,
            on=["unique_id", "ds_utc"],
            how="left",
        )
        if future_with_y["y"].isna().any():
            skipped.append({"forecast_origin_utc": origin, "reason": "missing future target"})
            continue

        for column in origin_frames.covariates:
            if column not in covariate_union:
                covariate_union.append(column)

        full = pd.concat(
            [origin_frames.history_full, future_with_y],
            ignore_index=True,
            sort=False,
        )
        full = normalize_covariate_dtypes(full, origin_frames.covariates)
        full["item_id"] = (
            full["unique_id"].astype(str)
            + "__origin_"
            + origin_frames.origin.strftime("%Y%m%dT%H%MZ")
        )
        full["timestamp"] = to_chronos_timestamp(full["ds_utc"])
        frame_columns = ["item_id", "timestamp", "y", *origin_frames.covariates]
        frames.append(full[frame_columns].rename(columns={"y": "target"}))

    if not frames:
        return pd.DataFrame(), covariate_union, pd.DataFrame(skipped)

    train_df = pd.concat(frames, ignore_index=True, sort=False)
    for column in covariate_union:
        if column not in train_df.columns:
            train_df[column] = pd.NA
    train_df = normalize_covariate_dtypes(train_df, covariate_union)
    train_df = train_df[["item_id", "timestamp", "target", *covariate_union]].sort_values(
        ["item_id", "timestamp"]
    )
    return train_df.reset_index(drop=True), covariate_union, pd.DataFrame(skipped)


lora_training_origins = choose_lora_training_origins(
    panel,
    origins,
    train_days=LORA_TRAIN_DAYS,
    max_origins=LORA_MAX_TRAIN_ORIGINS,
    at_hour_utc=AT_HOUR_UTC,
    min_history_days=MIN_HISTORY_DAYS,
)

lora_train_df, lora_covariates, lora_skipped = make_lora_training_frame(
    panel,
    lora_training_origins,
    context_length=LORA_CONTEXT_LENGTH,
    prediction_length=effective_lora_prediction_length,
    weather=weather_long,
)

{
    "candidate_training_origins": len(lora_training_origins),
    "effective_lora_prediction_length": effective_lora_prediction_length,
    "training_rows": len(lora_train_df),
    "training_items": 0 if lora_train_df.empty else lora_train_df["item_id"].nunique(),
    "known_future_covariates": len(lora_covariates),
    "skipped_origins": len(lora_skipped),
}

In [ ]:
display(lora_train_df.head())
display(lora_skipped.head(20))

## 8. LoRA Run

This cell fine-tunes a low-rank adapter for 300 steps, then evaluates the adapter on the same origins as the zero-shot runs.

In [ ]:
lora_predictions_df = pd.DataFrame()
lora_experiment = next(experiment for experiment in EXPERIMENTS if experiment.finetune_mode == "lora")

if RUN_LORA:
    if weather_long is None:
        raise FileNotFoundError(f"Missing {WEATHER_LONG_PATH}; LoRA experiment requires weather covariates.")
    if lora_train_df.empty:
        raise ValueError("No LoRA training rows were built. Inspect lora_skipped before training.")

    from chronos.chronos2 import preprocess

    train_inputs = preprocess.from_data_frame(
        lora_train_df,
        target_columns=["target"],
        prediction_length=effective_lora_prediction_length,
        id_column="item_id",
        timestamp_column="timestamp",
        known_covariates_names=lora_covariates,
    )

    lora_pipeline = pipeline.fit(
        inputs=train_inputs,
        prediction_length=effective_lora_prediction_length,
        context_length=LORA_CONTEXT_LENGTH,
        min_past=LORA_CONTEXT_LENGTH,
        num_steps=LORA_NUM_STEPS,
        learning_rate=LORA_LEARNING_RATE,
        batch_size=LORA_BATCH_SIZE,
        logging_steps=LORA_LOGGING_STEPS,
        finetune_mode="lora",
        **LORA_TRAINER_KWARGS,
    )

    lora_predictions_df = run_chronos_backtest(
        lora_pipeline,
        lora_experiment,
        panel,
        origins,
        weather=weather_long,
    )
else:
    print("RUN_LORA is False")

lora_predictions_df.shape

## 9. Evaluation Tables

The primary table is MAE/RMSE/bias by model and area, plus p10-p90 coverage and width when quantiles are available.

In [ ]:
chronos_prediction_frames = [frame for frame in [zero_shot_predictions_df, lora_predictions_df] if not frame.empty]
chronos_predictions_df = (
    pd.concat(chronos_prediction_frames, ignore_index=True)
    if chronos_prediction_frames
    else pd.DataFrame()
)
comparison_frames = [baseline_predictions]
if not chronos_predictions_df.empty:
    comparison_frames.append(chronos_predictions_df)
all_predictions = pd.concat(comparison_frames, ignore_index=True)
for optional_column in ["covariate_count", "chronos_prediction_length"]:
    if optional_column not in all_predictions.columns:
        all_predictions[optional_column] = pd.NA

if chronos_predictions_df.empty:
    print("No Chronos predictions have been generated yet; showing reference comparison scores.")

all_predictions = add_prediction_diagnostics(all_predictions)
scores = model_score_table(all_predictions)
probabilistic_scores = probabilistic_metric_table(all_predictions)
display(scores[scores["area"].eq("ALL")].sort_values("mae"))

{
    "chronos_prediction_rows": len(chronos_predictions_df),
    "comparison_prediction_rows": len(all_predictions),
}

In [ ]:
if not scores.empty:
    area_scores = scores[scores["area"].eq("ALL")].sort_values("mae")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=area_scores, y="model_label", x="mae", ax=ax, color="#4c78a8")
    ax.set_title("Chronos-2 experiments on shared day-ahead origins")
    ax.set_xlabel("MAE")
    ax.set_ylabel("")
    plt.show()

In [ ]:
if not all_predictions.empty:
    per_origin = (
        all_predictions.groupby(["forecast_origin_utc", "model_label"], dropna=False)
        .agg(
            rows=("y", "size"),
            mae=("abs_error", "mean"),
            bias=("error", "mean"),
            covariate_count=("covariate_count", "max"),
            chronos_prediction_length=("chronos_prediction_length", "max"),
        )
        .reset_index()
    )
    display(per_origin.head(20))

    fig, ax = plt.subplots(figsize=(11, 4))
    sns.lineplot(data=per_origin, x="forecast_origin_utc", y="mae", hue="model_label", marker="o", ax=ax)
    ax.set_title("Per-origin MAE")
    ax.set_xlabel("Forecast origin")
    ax.set_ylabel("MAE")
    plt.xticks(rotation=30, ha="right")
    plt.show()

## 10. Write Local Artifacts

Artifacts are written under `results/notebook_chronos2_experimental_v1/`, which is ignored runtime output in this repo.

In [ ]:
if WRITE_ARTIFACTS and not all_predictions.empty:
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    all_predictions.to_parquet(RESULT_DIR / "predictions.parquet", index=False)
    all_predictions.to_parquet(RESULT_DIR / "comparison_predictions.parquet", index=False)
    chronos_predictions_df.to_parquet(RESULT_DIR / "chronos_predictions.parquet", index=False)
    scores.to_parquet(RESULT_DIR / "model_scores.parquet", index=False)
    probabilistic_scores.to_parquet(RESULT_DIR / "probabilistic_metrics.parquet", index=False)

    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_id": MODEL_ID,
        "panel_path": str(PANEL_PATH),
        "weather_frame_path": str(WEATHER_FRAME_PATH),
        "weather_long_path": str(WEATHER_LONG_PATH),
        "result_dir": str(RESULT_DIR),
        "validation_arena": "compact_recent_weather",
        "validation_months": WEATHER_BACKTEST_MONTHS,
        "weather_frame_months": WEATHER_FRAME_MONTHS,
        "smoke_max_origins": SMOKE_MAX_ORIGINS,
        "origins": {
            "count": int(len(origins)),
            "min": origins["forecast_origin_utc"].min(),
            "max": origins["forecast_origin_utc"].max(),
            "at_hour_utc": AT_HOUR_UTC,
        },
        "baseline_model_count": int(baseline_predictions["model_label"].nunique()),
        "experiments": [asdict(experiment) for experiment in EXPERIMENTS],
        "run_flags": {
            "run_zero_shot": RUN_ZERO_SHOT,
            "run_lora": RUN_LORA,
            "cross_learning": CROSS_LEARNING,
            "weather_covariate_mode": WEATHER_COVARIATE_MODE,
            "add_weather_ensemble_features": ADD_WEATHER_ENSEMBLE_FEATURES,
            "add_weather_derived_features": ADD_WEATHER_DERIVED_FEATURES,
        },
        "lora": {
            "context_length": LORA_CONTEXT_LENGTH,
            "num_steps": LORA_NUM_STEPS,
            "batch_size": LORA_BATCH_SIZE,
            "learning_rate": LORA_LEARNING_RATE,
            "configured_prediction_length": LORA_PREDICTION_LENGTH,
            "effective_prediction_length": effective_lora_prediction_length,
            "training_items": 0 if lora_train_df.empty else int(lora_train_df["item_id"].nunique()),
            "known_future_covariates": len(lora_covariates),
            "skipped_origins": len(lora_skipped),
        },
    }
    (RESULT_DIR / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    print(f"Wrote Chronos-2 experiment artifacts to {RESULT_DIR}")
else:
    print("No artifacts written.")

## 11. Reading The Results

Useful first questions after execution:

- Does 1344 hours beat 336 hours for target-only zero-shot, or does older context add noise?
- Do calendar covariates improve the bridge-plus-delivery task once Chronos already sees hourly seasonality in the target?
- Do weather covariates improve aggregate MAE and per-origin stability, or only selected delivery days?
- Does the 300-step LoRA adapter beat the best zero-shot variant on the same origins, and does it do so without worsening interval coverage?
- Does any Chronos variant clear the production baseline hurdle in the same CatBoost weather arena?

Because the same development validation arena may be inspected repeatedly, treat the best Chronos number here as a selection signal. A production claim still needs a frozen promotion evaluation or a live shadow run.